# Classification

In [ ]:
import tensorflow as tf
import numpy as np
from utils.preprocessing import preprocess_image


# Load the classification model once
classification_model = tf.keras.models.load_model(
    "models/efficientnetB1.h5"
)

# Class labels corresponding to model outputs
CLASS_NAMES = [
    "Actinic keratoses",
    "Basal cell carcinoma",
    "Benign keratosis-like lesions",
    "Dermatofibroma",
    "Melanocytic nevi",
    "Melanoma",
    "Squamous cell carcinoma",
    "Vascular lesions"
]


def classify(image):
    """
    Classify a skin lesion image using the EfficientNet model.

    Parameters:
    image : Input image in BGR format (OpenCV)

    Returns:
    predicted_class : Name of the predicted lesion type
    confidence      : Prediction confidence score
    """

    # Preprocess image for EfficientNet
    input_tensor = preprocess_image(image)

    # Run model inference
    predictions = classification_model.predict(
        input_tensor, verbose=0
    )[0]

    # Get predicted class index
    predicted_index = np.argmax(predictions)

    return (
        CLASS_NAMES[predicted_index],
        float(predictions[predicted_index])
    )


# Gradcam

In [ ]:
import tensorflow as tf
import numpy as np
import cv2


def generate_gradcam(model, image, class_index):
    """
    Generate Grad-CAM heatmap for an EfficientNet-based classifier.

    Parameters:
    model       : Full classification model
    image       : Preprocessed image (1, H, W, 3)
    class_index : Index of predicted class
    """

    # --------------------------------------------------
    # Find EfficientNet backbone dynamically
    # --------------------------------------------------
    backbone = None
    for layer in model.layers:
        if "efficientnet" in layer.name.lower():
            backbone = layer
            break

    if backbone is None:
        raise ValueError("EfficientNet backbone not found in model.")

    # --------------------------------------------------
    # Get last convolutional layer inside EfficientNet
    # --------------------------------------------------
    last_conv_layer = backbone.get_layer("top_activation")

    # Model to extract feature maps
    feature_extractor = tf.keras.Model(
        inputs=backbone.input,
        outputs=last_conv_layer.output
    )

    # Layers after backbone (classifier head)
    classifier_layers = model.layers[
        model.layers.index(backbone) + 1 :
    ]

    with tf.GradientTape() as tape:
        conv_output = feature_extractor(image)
        tape.watch(conv_output)

        x = conv_output
        for layer in classifier_layers:
            x = layer(x)

        predictions = x
        loss = predictions[:, class_index]

    # --------------------------------------------------
    # Compute Grad-CAM
    # --------------------------------------------------
    gradients = tape.gradient(loss, conv_output)
    pooled_gradients = tf.reduce_mean(
        gradients, axis=(0, 1, 2)
    )

    conv_output = conv_output[0]
    heatmap = tf.reduce_sum(
        conv_output * pooled_gradients, axis=-1
    )

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy()


def overlay_gradcam(image, heatmap, alpha=0.4):
    """
    Overlay Grad-CAM heatmap on original image.
    """

    heatmap = cv2.resize(
        heatmap, (image.shape[1], image.shape[0])
    )

    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(
        heatmap, cv2.COLORMAP_JET
    )

    return cv2.addWeighted(
        image, 1 - alpha, heatmap, alpha, 0
    )


# Report_Generator

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# --------------------------------------------------
# Load embeddings and FAISS vector store
# --------------------------------------------------
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector_store = FAISS.load_local(
    "rag/vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)


# --------------------------------------------------
# Initialize local LLM (Ollama)
# --------------------------------------------------
llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0.3  # lower temperature for safer medical responses
)

output_parser = StrOutputParser()


# --------------------------------------------------
# Prompt templates
# --------------------------------------------------
patient_prompt = ChatPromptTemplate.from_template(
    """
    You are a compassionate medical assistant.

    Explain the diagnosed skin condition in simple, non-technical language
    so that a patient can easily understand it.

    Disease: {disease}
    Prediction confidence: {confidence}
    Lesion details: {lesion_info}

    Medical reference context:
    {context}

    Instructions:
    - Avoid medical jargon
    - Reassure the patient
    - Explain what the condition is
    - Mention general care advice
    - Clearly state that this is NOT a final diagnosis
    """
)

doctor_prompt = ChatPromptTemplate.from_template(
    """
    You are an experienced dermatologist preparing a clinical support report.

    Disease: {disease}
    Model confidence: {confidence}
    Lesion metrics: {lesion_info}

    Reference medical literature:
    {context}

    Instructions:
    - Use appropriate clinical terminology
    - Explain diagnostic reasoning
    - Mention possible differential diagnoses if relevant
    - Suggest next clinical steps (biopsy, dermoscopy, follow-up)
    - Clearly state that this is an AI-assisted opinion
    """
)


# --------------------------------------------------
# Report generation
# --------------------------------------------------
def generate_reports(disease, confidence, lesion_info):
    """
    Generate patient-friendly and doctor-focused reports using RAG.

    Parameters:
    disease      : Predicted disease name
    confidence   : Model confidence score
    lesion_info  : Information derived from segmentation

    Returns:
    patient_report : Simplified explanation for patients
    doctor_report  : Clinical explanation for doctors
    """

    # Retrieve relevant medical documents
    documents = vector_store.similarity_search(disease, k=3)
    context = "\n\n".join(doc.page_content for doc in documents)

    # Patient report
    patient_chain = patient_prompt | llm | output_parser
    patient_report = patient_chain.invoke({
        "disease": disease,
        "confidence": f"{confidence * 100:.2f}%",
        "lesion_info": lesion_info,
        "context": context
    })

    # Doctor report
    doctor_chain = doctor_prompt | llm | output_parser
    doctor_report = doctor_chain.invoke({
        "disease": disease,
        "confidence": f"{confidence * 100:.2f}%",
        "lesion_info": lesion_info,
        "context": context
    })

    return patient_report, doctor_report


# PDF_generator

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from datetime import datetime


def generate_pdf(file_path, title, content):
    """
    Generate a simple PDF report.

    Parameters:
    file_path : Path where the PDF will be saved
    title     : Title of the report
    content   : Text content of the report
    """

    # Create PDF canvas
    pdf = canvas.Canvas(file_path, pagesize=A4)
    page_width, page_height = A4

    # -------------------------------
    # Title
    # -------------------------------
    pdf.setFont("Helvetica-Bold", 16)
    pdf.drawString(50, page_height - 50, title)

    # -------------------------------
    # Timestamp
    # -------------------------------
    pdf.setFont("Helvetica", 10)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    pdf.drawString(50, page_height - 80, f"Generated on: {timestamp}")

    # -------------------------------
    # Body text
    # -------------------------------
    y_position = page_height - 120
    pdf.setFont("Helvetica", 11)

    for line in content.split("\n"):
        # Create a new page if space is insufficient
        if y_position < 50:
            pdf.showPage()
            pdf.setFont("Helvetica", 11)
            y_position = page_height - 50

        pdf.drawString(50, y_position, line)
        y_position -= 15

    # Save the PDF
    pdf.save()


# Preprocessing

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input

# Expected input size for EfficientNet
IMG_SIZE = (224, 224)


def preprocess_image(image):
    """
    Preprocess an image for EfficientNet classification.

    Parameters:
    image : Input image in BGR format (OpenCV)

    Returns:
    A NumPy array of shape (1, 224, 224, 3) ready for model inference
    """

    # Convert BGR to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize to model input size
    image = cv2.resize(image, IMG_SIZE)

    # Apply EfficientNet-specific preprocessing
    image = preprocess_input(image)

    # Add batch dimension
    return np.expand_dims(image, axis=0)


In [ ]:
# Segmentation

In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# Input size expected by the U-Net model
IMG_SIZE = (128, 128)

# Load the segmentation model once
segmentation_model = tf.keras.models.load_model(
    "models/unet_model(128).h5"
)


def segment_lesion(image):
    """
    Segment the lesion region from the input image using a U-Net model.

    Parameters:
    image : Input image in BGR format (OpenCV)

    Returns:
    segmented_image : Original image masked by the predicted lesion region
    mask            : Binary segmentation mask resized to original image size
    """

    # Resize image to U-Net input size and normalize
    resized_image = cv2.resize(image, IMG_SIZE)
    normalized_image = resized_image / 255.0

    # Add batch dimension
    input_tensor = np.expand_dims(normalized_image, axis=0)

    # Predict segmentation mask
    predicted_mask = segmentation_model.predict(input_tensor, verbose=0)[0]

    # Convert probability map to binary mask
    binary_mask = (predicted_mask > 0.5).astype(np.uint8)

    # Resize mask back to original image size
    binary_mask = cv2.resize(
        binary_mask,
        (image.shape[1], image.shape[0])
    )

    # Apply mask to the original image
    segmented_image = image * binary_mask[:, :, None]

    return segmented_image, binary_mask


# Streamlit

In [ ]:
import streamlit as st
import cv2
import numpy as np
import tensorflow as tf

from utils.segmentation import segment_lesion
from utils.classification import classify, CLASS_NAMES
from utils.report_generator import generate_reports
from utils.gradcam import generate_gradcam, overlay_gradcam
from utils.pdf_generator import generate_pdf
from utils.preprocessing import preprocess_image


# --------------------------------------------------
# Streamlit page configuration
# --------------------------------------------------
st.set_page_config(
    page_title="Skin Lesion Diagnosis",
    layout="wide"
)

st.title("Skin Lesion Diagnosis and Report Generation")


# --------------------------------------------------
# Load classification model only once
# --------------------------------------------------
@st.cache_resource
def load_classifier():
    return tf.keras.models.load_model("models/efficientnetB1.h5")

classifier_model = load_classifier()


# --------------------------------------------------
# Image upload
# --------------------------------------------------
uploaded_file = st.file_uploader(
    "Upload a skin lesion image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    # --------------------------------------------------
    # Read uploaded image using OpenCV
    # --------------------------------------------------
    image_bytes = np.frombuffer(uploaded_file.read(), np.uint8)
    image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

    st.image(
        image,
        caption="Original Image",
        width=350
    )


    # --------------------------------------------------
    # Lesion segmentation (for visualization and metrics)
    # --------------------------------------------------
    segmented_image, mask = segment_lesion(image)

    st.image(
        segmented_image,
        caption="Segmented Lesion",
        width=350
    )


    # --------------------------------------------------
    # Classification (always use original image)
    # --------------------------------------------------
    disease, confidence = classify(image)

    st.success(
        f"Prediction: {disease} ({confidence * 100:.2f}%)"
    )

    lesion_info = f"Lesion area (pixels): {int(mask.sum())}"


    # --------------------------------------------------
    # Grad-CAM explainability
    # --------------------------------------------------
    st.subheader("Model Explanation (Grad-CAM)")

    preprocessed_image = preprocess_image(image)
    class_index = CLASS_NAMES.index(disease)

    heatmap = generate_gradcam(
        model=classifier_model,
        image=preprocessed_image,
        class_index=class_index
    )

    gradcam_image = overlay_gradcam(image, heatmap)

    st.image(
        gradcam_image,
        caption="Grad-CAM Heatmap",
        width=350
    )


    # --------------------------------------------------
    # Report generation
    # --------------------------------------------------
    if st.button("Generate Reports"):

        with st.spinner("Generating reports..."):
            patient_report, doctor_report = generate_reports(
                disease,
                confidence,
                lesion_info
            )

        # --------------------------------------------------
        # Display reports
        # --------------------------------------------------
        st.subheader("Patient Report")
        st.write(patient_report)

        st.subheader("Doctor Report")
        st.write(doctor_report)


        # --------------------------------------------------
        # Generate PDF files
        # --------------------------------------------------
        generate_pdf(
            "outputs/patient_report.pdf",
            "Patient Medical Report",
            patient_report
        )

        generate_pdf(
            "outputs/doctor_report.pdf",
            "Doctor Clinical Report",
            doctor_report
        )


        # --------------------------------------------------
        # Download buttons
        # --------------------------------------------------
        col1, col2 = st.columns(2)

        with col1:
            with open("outputs/patient_report.pdf", "rb") as file:
                st.download_button(
                    label="Download Patient Report (PDF)",
                    data=file,
                    file_name="patient_report.pdf",
                    mime="application/pdf"
                )

        with col2:
            with open("outputs/doctor_report.pdf", "rb") as file:
                st.download_button(
                    label="Download Doctor Report (PDF)",
                    data=file,
                    file_name="doctor_report.pdf",
                    mime="application/pdf"
                )
